In [ ]:
import pandas as pd
from econml.dml import LinearDML, CausalForestDML
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

In [ ]:
df = pd.read_feather("../data/processed/df_export.feather").set_index("index")

In [ ]:
# 2. Define a single config dict
config = {
    # variables
    "X_cols": [
        "days_since",
        "paid_search",
        "paid_shopping",
        "email",
        "local",
        "program"
    ],
    "T_col": "program",      # pick whichever channel you want as 'treatment'
    "y_col": "ga",               # your outcome

    # first-stage learners (m(X), e(X))
    "model_y": RandomForestRegressor(n_estimators=100, random_state=0),
    "model_t": RandomForestRegressor(n_estimators=100, random_state=0),

    # second-stage for constant effect
    "model_final_const": LinearRegression(fit_intercept=False),

    # CV for cross-fitting
    "cv": KFold(n_splits=5, shuffle=True, random_state=0)
}

# 3. Slice your arrays once from config
X = df[config["X_cols"]]
T = df[config["T_col"]]
y = df[config["y_col"]]

In [ ]:
# 4a. Constant (average) effect estimator
est_const = LinearDML(
    model_y    = config["model_y"],
    model_t    = config["model_t"],
    discrete_treatment=False,
    cv = config["cv"]
)

est_const.fit(Y=y, T=T, X=X)
print("ATE (constant):", est_const.ate(X))

ATE (constant): -0.0050109149882311904


In [ ]:
# 4b. Non-parametric heterogeneous CATE
est_het = CausalForestDML(
    model_y = config["model_y"],
    model_t = config["model_t"],
    discrete_treatment=False,
    cv = config["cv"]
)
est_het.fit(Y=y, T=T, X=X)
cate = est_het.effect(X)
print("First 5 CATEs:", cate[:5])

First 5 CATEs: [0.00144399 0.00056891 0.00090079 0.00016814 0.00091016]
